# Transfer Learning และการเปรียบเทียบกับ CNN ที่ใช้ Random Initialization

เปรียบเทียบ ResNet18 Transfer Learning, EfficientNet-B0 Transfer Learning,
MobileNetV3-Small Transfer Learning และ Custom CNN (From Scratch) จากบทก่อนหน้า

ฝึกแต่ละ Model เป็นเวลา 10 Epochs ด้วย Dataset เดิม และใช้ Dense Classification Head
ขนาด 128 → ReLU → 3 เลือก Model จาก Validation Set แล้ว Export Model ที่ใช้ Transfer Learning เป็น ONNX


## การเตรียม Environment บน Colab

เลือก T4 GPU ในเมนู Runtime แล้วติดตั้ง Libraries เพิ่มเติม โดยคง PyTorch/CUDA ที่ Colab จัดเตรียมไว้


In [ ]:
%pip install -q torchinfo tqdm livelossplot ipywidgets onnx onnxscript onnxruntime

## การดาวน์โหลด Dataset จาก GitHub

ดาวน์โหลด ZIP จาก Week 10 และตรวจ SHA-256 ก่อนแตกไฟล์ ใช้ Train/Validation/Test Split เดิม
หากมี ZIP อยู่แล้วจะใช้ไฟล์เดิม การแตก ZIP ไม่เขียนทับภาพที่มีอยู่ หากต้องการเริ่มใหม่ให้ใช้ Runtime ใหม่
ข้อมูลใน `/content` ไม่ถาวร เมื่อ Runtime ถูกล้างให้ประมวลผลเซลล์นี้อีกครั้ง
แหล่งที่มาและเงื่อนไขการใช้ภาพอยู่ใน `Dataset/README.md`

In [ ]:
%cd /content
import hashlib
from pathlib import Path
from urllib.request import urlretrieve

DATASET_URL = 'https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W10_CNN_TransferLearning/food_pizza_steak_sushi_colab.zip'
DATASET_ZIP = Path('/content/food_pizza_steak_sushi_colab.zip')
EXPECTED_SHA256 = '8da3c2870ddff28cafcca222939bdbb1f909506c6e8f6833386e01eaef53b431'

if not DATASET_ZIP.is_file():
    urlretrieve(DATASET_URL, DATASET_ZIP)  # ดาวน์โหลด ZIP โดยไม่ต้อง Upload จากเครื่อง

actual_sha256 = hashlib.sha256(DATASET_ZIP.read_bytes()).hexdigest()
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError('Dataset ZIP: SHA-256 mismatch; ตรวจหรือลบ ZIP ที่ไม่สมบูรณ์แล้วดาวน์โหลดใหม่')

!unzip -q -n /content/food_pizza_steak_sushi_colab.zip -d /content
print('Dataset พร้อมใช้:', Path('/content/Dataset'))

## 0. การนำเข้า Libraries และกำหนด Device

ใช้ PyTorch และ Torchvision สำหรับสร้างและฝึก Model


In [ ]:
import copy
import json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)
from torchinfo import summary
from tqdm.auto import tqdm
from livelossplot import PlotLosses
from IPython.display import display

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(DEVICE)

## 1. การเตรียม Dataset

ใช้ Training Set 360 ภาพ Validation Set 90 ภาพ และ Test Set 150 ภาพจากโฟลเดอร์เดิม


In [ ]:
DATA_DIR = Path('Dataset')
raw_train = datasets.ImageFolder(DATA_DIR / 'train')  # ชื่อ Subfolder เป็น Class Label
raw_valid = datasets.ImageFolder(DATA_DIR / 'valid')
raw_test = datasets.ImageFolder(DATA_DIR / 'test')
CLASS_NAMES = raw_train.classes  # ลำดับ Class Names ต้องตรงกับ Output Index ของ Model
NUM_CLASSES = len(CLASS_NAMES)
assert raw_train.class_to_idx == raw_valid.class_to_idx == raw_test.class_to_idx  # ตรวจ Class Mapping ทุก Split
EPOCHS = 10
BATCH_SIZE = 32
loss_fn = nn.CrossEntropyLoss()

image = raw_valid[0][0]  # Dataset รายการแรก → (PIL Image, Label); [0] ตัวหลังเลือก Image ไม่ใช่ Batch
display(image)
print(CLASS_NAMES)

## ResNet18 Transfer Learning

โหลด Pretrained Weights จาก ImageNet และทำนายภาพก่อนปรับ Classification Head ผลลัพธ์ยังอ้างอิง 1,000 Classes ของ ImageNet


In [ ]:
torch.manual_seed(42)
resnet18_weights = ResNet18_Weights.DEFAULT  # ชุด ImageNet Pretrained Weights ที่ Torchvision แนะนำ
resnet18_model = resnet18(weights=resnet18_weights)  # โหลด Architecture พร้อม Pretrained Weights
resnet18_eval_transform = resnet18_weights.transforms()  # Resize/CenterCrop/Tensor/Normalize ให้ตรงกับ Weights
resnet18_model.eval()  # สาธิต Inference ก่อน Training; ใช้ Evaluation Mode และคง BatchNorm Statistics
with torch.no_grad():  # ไม่เก็บ Computational Graph เฉพาะ Block นี้; ไม่ได้ Freeze Model ถาวร
    # unsqueeze(0): [3,224,224] → [1,3,224,224]; softmax(1): คะแนน 1,000 Classes; [0]: ภาพแรก
    scores = resnet18_model(resnet18_eval_transform(image).unsqueeze(0)).softmax(1)[0]
# argmax เลือก Class Index; meta['categories'] แปลง Index เป็นชื่อ ImageNet Class
print('ImageNet:', resnet18_weights.meta['categories'][scores.argmax().item()])

### Freeze Backbone และปรับ Classification Head

Freeze Parameters ของ Backbone และฝึกเฉพาะ Head `512 → 128 → ReLU → 3` โดยส่งค่า Logits เข้า CrossEntropyLoss โดยตรง


In [ ]:
for parameter in resnet18_model.parameters():
    parameter.requires_grad = False  # Freeze Parameters เดิมก่อนสร้าง Head ใหม่

torch.manual_seed(42)
resnet18_model.fc = nn.Sequential(  # แทน Head เดิมทั้งชุด; Parameters ใหม่ฝึกได้ตามปกติ
    nn.Linear(512, 128),  # รับ Feature Vector จาก Backbone
    nn.ReLU(),
    nn.Linear(128, NUM_CLASSES),  # Output เป็น 3 Logits ไม่ใส่ Softmax ก่อน Loss
)
resnet18_model.eval()
summary(resnet18_model, input_size=(1, 3, 224, 224), device='cpu',
        col_names=['output_size', 'num_params', 'trainable'], depth=2)

### การเตรียมภาพและ DataLoader

ใช้ Transform ที่กำหนดสำหรับ Pretrained Weights นี้ และใช้ RandomHorizontalFlip เป็น Data Augmentation เฉพาะ Training Set


In [ ]:
resnet18_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Data Augmentation เฉพาะ Training Set
    resnet18_eval_transform,
])
resnet18_train_set = datasets.ImageFolder(DATA_DIR / 'train', transform=resnet18_train_transform)
resnet18_val_set = datasets.ImageFolder(DATA_DIR / 'valid', transform=resnet18_eval_transform)
resnet18_train_loader = DataLoader(resnet18_train_set, batch_size=BATCH_SIZE, shuffle=True)  # สุ่มลำดับทุก Epoch
resnet18_val_loader = DataLoader(resnet18_val_set, batch_size=BATCH_SIZE, shuffle=False)  # Validation ไม่สุ่มภาพ

### การฝึก ResNet18 Classification Head

`eval()` คง BatchNorm Statistics ของ Backbone ส่วน `head.train()` กำหนด Training Mode ของ Head เก็บ Weights จาก Epoch ที่ Validation Loss ต่ำสุด


In [ ]:
torch.manual_seed(42)
resnet18_model = resnet18_model.to(DEVICE)
optimizer = torch.optim.Adam(resnet18_model.fc.parameters(), lr=0.001)  # ปรับเฉพาะ Parameters ที่ส่งให้ Optimizer
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
live_plot = PlotLosses()
best_loss = float('inf')  # เริ่มต้นเพื่อให้บันทึก Weights จาก Epoch แรกได้

for epoch in range(EPOCHS):
    resnet18_model.eval()  # คง BatchNorm Statistics ของ Backbone; ไม่ได้ปิด Gradient
    resnet18_model.fc.train()  # Training Mode เฉพาะ Head
    total_loss, total_correct = 0.0, 0
    for images, labels in tqdm(resnet18_train_loader, desc=f'Epoch {epoch + 1}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()  # ล้าง Gradient ที่สะสมจาก Batch ก่อนหน้า
        logits = resnet18_model(images)  # Forward Pass: ได้ Logits ของ 3 Classes
        loss = loss_fn(logits, labels)  # CrossEntropyLoss รับ Logits โดยไม่ผ่าน Softmax
        loss.backward()  # Backpropagation เฉพาะ Parameters ที่ requires_grad=True
        optimizer.step()  # ปรับ Weights ด้วย Gradient
        total_loss += loss.item() * len(labels)  # สะสมตามจำนวนภาพ รวม Batch สุดท้ายที่อาจเล็กกว่า
        total_correct += (logits.argmax(1) == labels).sum().item()  # นับ Prediction ที่ถูกต้อง
    train_loss = total_loss / len(resnet18_train_set)
    train_acc = total_correct / len(resnet18_train_set)

    resnet18_model.eval()  # Validation: ใช้ Evaluation Mode ทุก Layer
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():  # Validation ไม่ต้องเก็บ Computational Graph สำหรับ Gradient
        for images, labels in resnet18_val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = resnet18_model(images)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            total_correct += (logits.argmax(1) == labels).sum().item()
    val_loss = total_loss / len(resnet18_val_set)
    val_acc = total_correct / len(resnet18_val_set)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    live_plot.update({'loss': train_loss, 'val_loss': val_loss,
                      'accuracy': train_acc, 'val_accuracy': val_acc})
    live_plot.send()
    if val_loss < best_loss:  # เลือก Best Weights ด้วย Validation Loss ไม่ใช่ Training Loss
        best_loss = val_loss
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = copy.deepcopy(resnet18_model.state_dict())  # สำเนาอิสระ ไม่เปลี่ยนตาม Training รอบถัดไป

resnet18_model.load_state_dict(best_state)  # คืน Best Weights แทนการใช้ Weights จาก Epoch สุดท้าย
resnet18_model.cpu().eval()  # ย้ายกลับ CPU เพื่อไม่เก็บทุก Model บน GPU พร้อมกัน
resnet18_result = {'Model': 'ResNet18', 'Pretrained': True, 'Input': 224,
          'Best epoch': best_epoch, 'Val loss': best_loss, 'Val accuracy': best_acc,
          'Trainable params': sum(p.numel() for p in resnet18_model.parameters() if p.requires_grad)}
resnet18_history = history
del optimizer, best_state
print(resnet18_result)

## EfficientNet-B0 Transfer Learning

โหลด Pretrained Weights จาก ImageNet และทำนายภาพก่อนปรับ Classification Head ผลลัพธ์ยังอ้างอิง 1,000 Classes ของ ImageNet


In [ ]:
torch.manual_seed(42)
efficientnet_b0_weights = EfficientNet_B0_Weights.DEFAULT  # ชุด ImageNet Pretrained Weights ที่ Torchvision แนะนำ
efficientnet_b0_model = efficientnet_b0(weights=efficientnet_b0_weights)  # โหลด Architecture พร้อม Pretrained Weights
efficientnet_b0_eval_transform = efficientnet_b0_weights.transforms()  # Resize/CenterCrop/Tensor/Normalize ให้ตรงกับ Weights
efficientnet_b0_model.eval()  # สาธิต Inference ก่อน Training; ใช้ Evaluation Mode และคง BatchNorm Statistics
with torch.no_grad():  # ไม่เก็บ Computational Graph เฉพาะ Block นี้; ไม่ได้ Freeze Model ถาวร
    # unsqueeze(0): [3,224,224] → [1,3,224,224]; softmax(1): คะแนน 1,000 Classes; [0]: ภาพแรก
    scores = efficientnet_b0_model(efficientnet_b0_eval_transform(image).unsqueeze(0)).softmax(1)[0]
# argmax เลือก Class Index; meta['categories'] แปลง Index เป็นชื่อ ImageNet Class
print('ImageNet:', efficientnet_b0_weights.meta['categories'][scores.argmax().item()])

### Freeze Backbone และปรับ Classification Head

Freeze Parameters ของ Backbone และฝึกเฉพาะ Head `1280 → 128 → ReLU → 3` โดยส่งค่า Logits เข้า CrossEntropyLoss โดยตรง


In [ ]:
for parameter in efficientnet_b0_model.parameters():
    parameter.requires_grad = False  # Freeze Parameters เดิมก่อนสร้าง Head ใหม่

torch.manual_seed(42)
efficientnet_b0_model.classifier = nn.Sequential(  # แทน Head เดิมทั้งชุด; Parameters ใหม่ฝึกได้ตามปกติ
    nn.Linear(1280, 128),  # รับ Feature Vector จาก Backbone
    nn.ReLU(),
    nn.Linear(128, NUM_CLASSES),  # Output เป็น 3 Logits ไม่ใส่ Softmax ก่อน Loss
)
efficientnet_b0_model.eval()
summary(efficientnet_b0_model, input_size=(1, 3, 224, 224), device='cpu',
        col_names=['output_size', 'num_params', 'trainable'], depth=2)

### การเตรียมภาพและ DataLoader

ใช้ Transform ที่กำหนดสำหรับ Pretrained Weights นี้ และใช้ RandomHorizontalFlip เป็น Data Augmentation เฉพาะ Training Set


In [ ]:
efficientnet_b0_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Data Augmentation เฉพาะ Training Set
    efficientnet_b0_eval_transform,
])
efficientnet_b0_train_set = datasets.ImageFolder(DATA_DIR / 'train', transform=efficientnet_b0_train_transform)
efficientnet_b0_val_set = datasets.ImageFolder(DATA_DIR / 'valid', transform=efficientnet_b0_eval_transform)
efficientnet_b0_train_loader = DataLoader(efficientnet_b0_train_set, batch_size=BATCH_SIZE, shuffle=True)  # สุ่มลำดับทุก Epoch
efficientnet_b0_val_loader = DataLoader(efficientnet_b0_val_set, batch_size=BATCH_SIZE, shuffle=False)  # Validation ไม่สุ่มภาพ

### การฝึก EfficientNet-B0 Classification Head

`eval()` คง BatchNorm Statistics ของ Backbone ส่วน `head.train()` กำหนด Training Mode ของ Head เก็บ Weights จาก Epoch ที่ Validation Loss ต่ำสุด


In [ ]:
torch.manual_seed(42)
efficientnet_b0_model = efficientnet_b0_model.to(DEVICE)
optimizer = torch.optim.Adam(efficientnet_b0_model.classifier.parameters(), lr=0.001)  # ปรับเฉพาะ Parameters ที่ส่งให้ Optimizer
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
live_plot = PlotLosses()
best_loss = float('inf')  # เริ่มต้นเพื่อให้บันทึก Weights จาก Epoch แรกได้

for epoch in range(EPOCHS):
    efficientnet_b0_model.eval()  # คง BatchNorm Statistics ของ Backbone; ไม่ได้ปิด Gradient
    efficientnet_b0_model.classifier.train()  # Training Mode เฉพาะ Head
    total_loss, total_correct = 0.0, 0
    for images, labels in tqdm(efficientnet_b0_train_loader, desc=f'Epoch {epoch + 1}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()  # ล้าง Gradient ที่สะสมจาก Batch ก่อนหน้า
        logits = efficientnet_b0_model(images)  # Forward Pass: ได้ Logits ของ 3 Classes
        loss = loss_fn(logits, labels)  # CrossEntropyLoss รับ Logits โดยไม่ผ่าน Softmax
        loss.backward()  # Backpropagation เฉพาะ Parameters ที่ requires_grad=True
        optimizer.step()  # ปรับ Weights ด้วย Gradient
        total_loss += loss.item() * len(labels)  # สะสมตามจำนวนภาพ รวม Batch สุดท้ายที่อาจเล็กกว่า
        total_correct += (logits.argmax(1) == labels).sum().item()  # นับ Prediction ที่ถูกต้อง
    train_loss = total_loss / len(efficientnet_b0_train_set)
    train_acc = total_correct / len(efficientnet_b0_train_set)

    efficientnet_b0_model.eval()  # Validation: ใช้ Evaluation Mode ทุก Layer
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():  # Validation ไม่ต้องเก็บ Computational Graph สำหรับ Gradient
        for images, labels in efficientnet_b0_val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = efficientnet_b0_model(images)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            total_correct += (logits.argmax(1) == labels).sum().item()
    val_loss = total_loss / len(efficientnet_b0_val_set)
    val_acc = total_correct / len(efficientnet_b0_val_set)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    live_plot.update({'loss': train_loss, 'val_loss': val_loss,
                      'accuracy': train_acc, 'val_accuracy': val_acc})
    live_plot.send()
    if val_loss < best_loss:  # เลือก Best Weights ด้วย Validation Loss ไม่ใช่ Training Loss
        best_loss = val_loss
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = copy.deepcopy(efficientnet_b0_model.state_dict())  # สำเนาอิสระ ไม่เปลี่ยนตาม Training รอบถัดไป

efficientnet_b0_model.load_state_dict(best_state)  # คืน Best Weights แทนการใช้ Weights จาก Epoch สุดท้าย
efficientnet_b0_model.cpu().eval()  # ย้ายกลับ CPU เพื่อไม่เก็บทุก Model บน GPU พร้อมกัน
efficientnet_b0_result = {'Model': 'EfficientNet-B0', 'Pretrained': True, 'Input': 224,
          'Best epoch': best_epoch, 'Val loss': best_loss, 'Val accuracy': best_acc,
          'Trainable params': sum(p.numel() for p in efficientnet_b0_model.parameters() if p.requires_grad)}
efficientnet_b0_history = history
del optimizer, best_state
print(efficientnet_b0_result)

## MobileNetV3-Small Transfer Learning

โหลด Pretrained Weights จาก ImageNet และทำนายภาพก่อนปรับ Classification Head ผลลัพธ์ยังอ้างอิง 1,000 Classes ของ ImageNet


In [ ]:
torch.manual_seed(42)
mobilenet_v3_small_weights = MobileNet_V3_Small_Weights.DEFAULT  # ชุด ImageNet Pretrained Weights ที่ Torchvision แนะนำ
mobilenet_v3_small_model = mobilenet_v3_small(weights=mobilenet_v3_small_weights)  # โหลด Architecture พร้อม Pretrained Weights
mobilenet_v3_small_eval_transform = mobilenet_v3_small_weights.transforms()  # Resize/CenterCrop/Tensor/Normalize ให้ตรงกับ Weights
mobilenet_v3_small_model.eval()  # สาธิต Inference ก่อน Training; ใช้ Evaluation Mode และคง BatchNorm Statistics
with torch.no_grad():  # ไม่เก็บ Computational Graph เฉพาะ Block นี้; ไม่ได้ Freeze Model ถาวร
    # unsqueeze(0): [3,224,224] → [1,3,224,224]; softmax(1): คะแนน 1,000 Classes; [0]: ภาพแรก
    scores = mobilenet_v3_small_model(mobilenet_v3_small_eval_transform(image).unsqueeze(0)).softmax(1)[0]
# argmax เลือก Class Index; meta['categories'] แปลง Index เป็นชื่อ ImageNet Class
print('ImageNet:', mobilenet_v3_small_weights.meta['categories'][scores.argmax().item()])

### Freeze Backbone และปรับ Classification Head

Freeze Parameters ของ Backbone และฝึกเฉพาะ Head `576 → 128 → ReLU → 3` โดยส่งค่า Logits เข้า CrossEntropyLoss โดยตรง


In [ ]:
for parameter in mobilenet_v3_small_model.parameters():
    parameter.requires_grad = False  # Freeze Parameters เดิมก่อนสร้าง Head ใหม่

torch.manual_seed(42)
mobilenet_v3_small_model.classifier = nn.Sequential(  # แทน Head เดิมทั้งชุด; Parameters ใหม่ฝึกได้ตามปกติ
    nn.Linear(576, 128),  # รับ Feature Vector จาก Backbone
    nn.ReLU(),
    nn.Linear(128, NUM_CLASSES),  # Output เป็น 3 Logits ไม่ใส่ Softmax ก่อน Loss
)
mobilenet_v3_small_model.eval()
summary(mobilenet_v3_small_model, input_size=(1, 3, 224, 224), device='cpu',
        col_names=['output_size', 'num_params', 'trainable'], depth=2)

### การเตรียมภาพและ DataLoader

ใช้ Transform ที่กำหนดสำหรับ Pretrained Weights นี้ และใช้ RandomHorizontalFlip เป็น Data Augmentation เฉพาะ Training Set


In [ ]:
mobilenet_v3_small_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # Data Augmentation เฉพาะ Training Set
    mobilenet_v3_small_eval_transform,
])
mobilenet_v3_small_train_set = datasets.ImageFolder(DATA_DIR / 'train', transform=mobilenet_v3_small_train_transform)
mobilenet_v3_small_val_set = datasets.ImageFolder(DATA_DIR / 'valid', transform=mobilenet_v3_small_eval_transform)
mobilenet_v3_small_train_loader = DataLoader(mobilenet_v3_small_train_set, batch_size=BATCH_SIZE, shuffle=True)  # สุ่มลำดับทุก Epoch
mobilenet_v3_small_val_loader = DataLoader(mobilenet_v3_small_val_set, batch_size=BATCH_SIZE, shuffle=False)  # Validation ไม่สุ่มภาพ

### การฝึก MobileNetV3-Small Classification Head

`eval()` คง BatchNorm Statistics ของ Backbone ส่วน `head.train()` กำหนด Training Mode ของ Head เก็บ Weights จาก Epoch ที่ Validation Loss ต่ำสุด


In [ ]:
torch.manual_seed(42)
mobilenet_v3_small_model = mobilenet_v3_small_model.to(DEVICE)
optimizer = torch.optim.Adam(mobilenet_v3_small_model.classifier.parameters(), lr=0.001)  # ปรับเฉพาะ Parameters ที่ส่งให้ Optimizer
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
live_plot = PlotLosses()
best_loss = float('inf')  # เริ่มต้นเพื่อให้บันทึก Weights จาก Epoch แรกได้

for epoch in range(EPOCHS):
    mobilenet_v3_small_model.eval()  # คง BatchNorm Statistics ของ Backbone; ไม่ได้ปิด Gradient
    mobilenet_v3_small_model.classifier.train()  # Training Mode เฉพาะ Head
    total_loss, total_correct = 0.0, 0
    for images, labels in tqdm(mobilenet_v3_small_train_loader, desc=f'Epoch {epoch + 1}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()  # ล้าง Gradient ที่สะสมจาก Batch ก่อนหน้า
        logits = mobilenet_v3_small_model(images)  # Forward Pass: ได้ Logits ของ 3 Classes
        loss = loss_fn(logits, labels)  # CrossEntropyLoss รับ Logits โดยไม่ผ่าน Softmax
        loss.backward()  # Backpropagation เฉพาะ Parameters ที่ requires_grad=True
        optimizer.step()  # ปรับ Weights ด้วย Gradient
        total_loss += loss.item() * len(labels)  # สะสมตามจำนวนภาพ รวม Batch สุดท้ายที่อาจเล็กกว่า
        total_correct += (logits.argmax(1) == labels).sum().item()  # นับ Prediction ที่ถูกต้อง
    train_loss = total_loss / len(mobilenet_v3_small_train_set)
    train_acc = total_correct / len(mobilenet_v3_small_train_set)

    mobilenet_v3_small_model.eval()  # Validation: ใช้ Evaluation Mode ทุก Layer
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():  # Validation ไม่ต้องเก็บ Computational Graph สำหรับ Gradient
        for images, labels in mobilenet_v3_small_val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = mobilenet_v3_small_model(images)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            total_correct += (logits.argmax(1) == labels).sum().item()
    val_loss = total_loss / len(mobilenet_v3_small_val_set)
    val_acc = total_correct / len(mobilenet_v3_small_val_set)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    live_plot.update({'loss': train_loss, 'val_loss': val_loss,
                      'accuracy': train_acc, 'val_accuracy': val_acc})
    live_plot.send()
    if val_loss < best_loss:  # เลือก Best Weights ด้วย Validation Loss ไม่ใช่ Training Loss
        best_loss = val_loss
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = copy.deepcopy(mobilenet_v3_small_model.state_dict())  # สำเนาอิสระ ไม่เปลี่ยนตาม Training รอบถัดไป

mobilenet_v3_small_model.load_state_dict(best_state)  # คืน Best Weights แทนการใช้ Weights จาก Epoch สุดท้าย
mobilenet_v3_small_model.cpu().eval()  # ย้ายกลับ CPU เพื่อไม่เก็บทุก Model บน GPU พร้อมกัน
mobilenet_v3_small_result = {'Model': 'MobileNetV3-Small', 'Pretrained': True, 'Input': 224,
          'Best epoch': best_epoch, 'Val loss': best_loss, 'Val accuracy': best_acc,
          'Trainable params': sum(p.numel() for p in mobilenet_v3_small_model.parameters() if p.requires_grad)}
mobilenet_v3_small_history = history
del optimizer, best_state
print(mobilenet_v3_small_result)

## Custom CNN (From Scratch)

ใช้ CNN Architecture จากบทก่อนหน้า กำหนด Weights ด้วย Random Initialization และฝึกทุก Layer โดยไม่ใช้ Transfer Learning

![CNN Architecture จากบทก่อนหน้า](https://raw.githubusercontent.com/vasuprint/AI-CE564/main/weeks/W10_CNN_TransferLearning/assets/food_cnn_architecture.png)


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)  # RGB 3 Channels → 16 Feature Maps
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)  # 16 → 32 Feature Maps
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(32 * 16 * 16, 128),  # Pool สองครั้ง: 64 → 32 → 16; Flatten ได้ 8192 Features
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        return self.classifier(self.flatten(x))

torch.manual_seed(42)
custom_cnn_model = SimpleCNN(NUM_CLASSES)  # Random Initialization; ฝึกทุก Layer โดยไม่ใช้ Pretrained Weights
summary(custom_cnn_model, input_size=(1, 3, 64, 64), device='cpu',
        col_names=['output_size', 'num_params', 'trainable'])

### การเตรียมภาพขนาด 64×64 และ Data Augmentation

ใช้ Split เดียวกับ Pretrained Models ทั้งสาม และคง Preprocessing จากบท CNN ก่อนหน้า


In [ ]:
custom_cnn_eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
custom_cnn_train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
custom_cnn_train_set = datasets.ImageFolder(DATA_DIR / 'train', transform=custom_cnn_train_transform)
custom_cnn_val_set = datasets.ImageFolder(DATA_DIR / 'valid', transform=custom_cnn_eval_transform)
custom_cnn_train_loader = DataLoader(custom_cnn_train_set, batch_size=BATCH_SIZE, shuffle=True)
custom_cnn_val_loader = DataLoader(custom_cnn_val_set, batch_size=BATCH_SIZE, shuffle=False)

### การฝึก Custom CNN

ปรับ Parameters ทุก Layer ส่วน Pretrained Models ทั้งสามปรับเฉพาะ Parameters ของ Head


In [ ]:
torch.manual_seed(42)
custom_cnn_model = custom_cnn_model.to(DEVICE)
optimizer = torch.optim.Adam(custom_cnn_model.parameters(), lr=0.001)  # ปรับเฉพาะ Parameters ที่ส่งให้ Optimizer
history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
live_plot = PlotLosses()
best_loss = float('inf')  # เริ่มต้นเพื่อให้บันทึก Weights จาก Epoch แรกได้

for epoch in range(EPOCHS):
    custom_cnn_model.train()
    total_loss, total_correct = 0.0, 0
    for images, labels in tqdm(custom_cnn_train_loader, desc=f'Epoch {epoch + 1}', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()  # ล้าง Gradient ที่สะสมจาก Batch ก่อนหน้า
        logits = custom_cnn_model(images)  # Forward Pass: ได้ Logits ของ 3 Classes
        loss = loss_fn(logits, labels)  # CrossEntropyLoss รับ Logits โดยไม่ผ่าน Softmax
        loss.backward()  # Backpropagation เฉพาะ Parameters ที่ requires_grad=True
        optimizer.step()  # ปรับ Weights ด้วย Gradient
        total_loss += loss.item() * len(labels)  # สะสมตามจำนวนภาพ รวม Batch สุดท้ายที่อาจเล็กกว่า
        total_correct += (logits.argmax(1) == labels).sum().item()  # นับ Prediction ที่ถูกต้อง
    train_loss = total_loss / len(custom_cnn_train_set)
    train_acc = total_correct / len(custom_cnn_train_set)

    custom_cnn_model.eval()  # Validation: ใช้ Evaluation Mode ทุก Layer
    total_loss, total_correct = 0.0, 0
    with torch.no_grad():  # Validation ไม่ต้องเก็บ Computational Graph สำหรับ Gradient
        for images, labels in custom_cnn_val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = custom_cnn_model(images)
            total_loss += loss_fn(logits, labels).item() * len(labels)
            total_correct += (logits.argmax(1) == labels).sum().item()
    val_loss = total_loss / len(custom_cnn_val_set)
    val_acc = total_correct / len(custom_cnn_val_set)

    history['loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['accuracy'].append(train_acc)
    history['val_accuracy'].append(val_acc)
    live_plot.update({'loss': train_loss, 'val_loss': val_loss,
                      'accuracy': train_acc, 'val_accuracy': val_acc})
    live_plot.send()
    if val_loss < best_loss:  # เลือก Best Weights ด้วย Validation Loss ไม่ใช่ Training Loss
        best_loss = val_loss
        best_acc = val_acc
        best_epoch = epoch + 1
        best_state = copy.deepcopy(custom_cnn_model.state_dict())  # สำเนาอิสระ ไม่เปลี่ยนตาม Training รอบถัดไป

custom_cnn_model.load_state_dict(best_state)  # คืน Best Weights แทนการใช้ Weights จาก Epoch สุดท้าย
custom_cnn_model.cpu().eval()  # ย้ายกลับ CPU เพื่อไม่เก็บทุก Model บน GPU พร้อมกัน
custom_cnn_result = {'Model': 'Custom CNN (From Scratch)', 'Pretrained': False, 'Input': 64,
          'Best epoch': best_epoch, 'Val loss': best_loss, 'Val accuracy': best_acc,
          'Trainable params': sum(p.numel() for p in custom_cnn_model.parameters() if p.requires_grad)}
custom_cnn_history = history
del optimizer, best_state
print(custom_cnn_result)

## 2. การเปรียบเทียบผลบน Validation Set

Model ทั้งสี่ใช้ Split และจำนวน Epochs เท่ากัน แต่ต่างกันด้าน Input Size, Data Augmentation และ ImageNet Pretraining จึงไม่สามารถแยกผลของ Architecture เพียงปัจจัยเดียวได้


In [ ]:
comparison = pd.DataFrame([resnet18_result, efficientnet_b0_result, mobilenet_v3_small_result, custom_cnn_result])
display(comparison.round(4))

### การเลือก Transfer Learning Model สำหรับ Deployment

เลือกจาก Validation Loss ต่ำสุดของ ResNet18, EfficientNet-B0 และ MobileNetV3-Small โดยไม่ใช้ Test Set ประกอบการเลือก


In [ ]:
best_transfer = comparison.iloc[:3].sort_values('Val loss').iloc[0]['Model']  # เลือกเฉพาะ 3 Pretrained Models ด้วย Validation
if best_transfer == 'ResNet18':
    selected_model = resnet18_model
    selected_transform = resnet18_eval_transform
elif best_transfer == 'EfficientNet-B0':
    selected_model = efficientnet_b0_model
    selected_transform = efficientnet_b0_eval_transform
else:
    selected_model = mobilenet_v3_small_model
    selected_transform = mobilenet_v3_small_eval_transform
print('Deploy:', best_transfer)

## 3. การประเมินบน Test Set: Transfer Learning Model

ประเมินหลังสิ้นสุด Model Selection และ Hyperparameter Tuning การใช้ Run All จะประเมิน Test Set ด้วย ห้ามนำผลกลับไปปรับ Model


In [ ]:
test_set_transfer = datasets.ImageFolder(DATA_DIR / 'test', transform=selected_transform)
test_loader_transfer = DataLoader(test_set_transfer, batch_size=BATCH_SIZE, shuffle=False)
selected_model.to(DEVICE).eval()  # Final Test ใช้ Best Weights ใน Evaluation Mode
total_loss, total_correct = 0.0, 0
with torch.no_grad():  # ประเมินอย่างเดียว ไม่มี Backpropagation หรือ Optimizer Step
    for images, labels in test_loader_transfer:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = selected_model(images)
        total_loss += loss_fn(logits, labels).item() * len(labels)
        total_correct += (logits.argmax(1) == labels).sum().item()
transfer_test = {'Test loss': total_loss / len(test_set_transfer),
                 'Test accuracy': total_correct / len(test_set_transfer)}
selected_model.cpu().eval()
print(transfer_test)

### การประเมินบน Test Set: Custom CNN (From Scratch)

ประเมินหลังสิ้นสุด Model Selection และ Hyperparameter Tuning การใช้ Run All จะประเมิน Test Set ด้วย ห้ามนำผลกลับไปปรับ Model


In [ ]:
test_set_scratch = datasets.ImageFolder(DATA_DIR / 'test', transform=custom_cnn_eval_transform)
test_loader_scratch = DataLoader(test_set_scratch, batch_size=BATCH_SIZE, shuffle=False)
custom_cnn_model.to(DEVICE).eval()  # Final Test ใช้ Best Weights ใน Evaluation Mode
total_loss, total_correct = 0.0, 0
with torch.no_grad():  # ประเมินอย่างเดียว ไม่มี Backpropagation หรือ Optimizer Step
    for images, labels in test_loader_scratch:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = custom_cnn_model(images)
        total_loss += loss_fn(logits, labels).item() * len(labels)
        total_correct += (logits.argmax(1) == labels).sum().item()
scratch_test = {'Test loss': total_loss / len(test_set_scratch),
                 'Test accuracy': total_correct / len(test_set_scratch)}
custom_cnn_model.cpu().eval()
print(scratch_test)

### การเปรียบเทียบผล Final Evaluation

เปรียบเทียบผลบน Test Set หลังเลือก Model ด้วย Validation Set แล้ว


In [ ]:
test_comparison = pd.DataFrame([
    {'Model': best_transfer, **transfer_test},
    {'Model': 'Custom CNN (From Scratch)', **scratch_test},
])
display(test_comparison.round(4))

## 4. การบันทึกและโหลด Model Weights

บันทึก Weights จาก Epoch ที่ Validation Loss ต่ำสุดของ Model ที่เลือก แล้วโหลดกลับเข้าสู่ Architecture เดิม


In [ ]:
OUT_DIR = Path('transfer_outputs')
OUT_DIR.mkdir(exist_ok=True)
WEIGHTS_PATH = OUT_DIR / 'food_transfer_weights.pt'
torch.save(selected_model.state_dict(), WEIGHTS_PATH)  # บันทึก Parameters/Buffer ไม่รวม Architecture หรือ Preprocessing

loaded_model = copy.deepcopy(selected_model).cpu()  # ใช้ Architecture เดิมที่ยังอยู่ใน Notebook
loaded_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True))  # โหลดเฉพาะไฟล์ที่เชื่อถือได้
loaded_model.eval()
with torch.no_grad():
    check_input = selected_transform(image).unsqueeze(0)
    torch.testing.assert_close(loaded_model(check_input), selected_model(check_input))
print('Save/Load OK')

### การบันทึก Preprocessing Configuration และ Class Names

ใช้ JSON ร่วมกับ ONNX เพื่อระบุ Preprocessing และ Class Order ซึ่งไม่ได้รวมอยู่ใน Model Graph นี้


In [ ]:
config = {
    'architecture': best_transfer, 'class_names': CLASS_NAMES,
    'resize_size': selected_transform.resize_size[0],
    'crop_size': selected_transform.crop_size[0],
    'mean': selected_transform.mean, 'std': selected_transform.std,
    'interpolation': selected_transform.interpolation.value,
}
CONFIG_PATH = OUT_DIR / 'food_transfer_config.json'
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding='utf-8')
comparison.to_json(OUT_DIR / 'validation_comparison.json', orient='records', indent=2)
test_comparison.to_json(OUT_DIR / 'test_comparison.json', orient='records', indent=2)

## 5. การ Export Model เป็น ONNX

กำหนด Input Tensor ชนิด float32 ขนาดคงที่ `[1, 3, 224, 224]` สำหรับ Inference ครั้งละหนึ่งภาพ


In [ ]:
import onnx

ONNX_PATH = OUT_DIR / 'food_transfer.onnx'
torch.onnx.export(  # Export Best Weights ใน Evaluation Mode ไม่รวม Preprocessing
    loaded_model, (torch.zeros(1, 3, 224, 224),), str(ONNX_PATH),
    input_names=['images'], output_names=['logits'],  # ชื่อต้องตรงกับ session.run()
    dynamo=True, opset_version=18, external_data=False,  # Exporter รุ่นใหม่; รวม Weights ใน ONNX ไฟล์เดียว
)
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))  # ตรวจโครงสร้าง ONNX; ตรวจตัวเลขอีกครั้งใน Parity Cell
print(ONNX_PATH)

## 6. Inference ด้วย ONNX Runtime

เซลล์นี้ทำงานโดยไม่ใช้ PyTorch สามารถกำหนด IMAGE_PATH เป็นภาพ JPEG/PNG ที่ต้องการทำนายได้


In [ ]:
import json
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
import onnxruntime as ort

CONFIG_PATH = Path('transfer_outputs/food_transfer_config.json')
ONNX_PATH = Path('transfer_outputs/food_transfer.onnx')
IMAGE_PATH = sorted(Path('Dataset/valid/pizza').glob('*.jpg'))[0]
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))  # ใช้ Preprocessing และ Class Order เดียวกับตอนฝึก
with Image.open(IMAGE_PATH) as source:
    image = ImageOps.exif_transpose(source).convert('RGB')  # แก้ Orientation จาก EXIF และกำหนด 3 Channels

# Resize ด้านสั้น → CenterCrop ให้ตรงกับ Torchvision
size = config['resize_size']
width, height = image.size
if width <= height:
    new_size = (size, int(size * height / width))
else:
    new_size = (int(size * width / height), size)
interpolation = {'bilinear': Image.Resampling.BILINEAR,
                 'bicubic': Image.Resampling.BICUBIC}[config['interpolation']]
resized = image.resize(new_size, resample=interpolation)
crop = config['crop_size']
left = round((resized.width - crop) / 2)
top = round((resized.height - crop) / 2)
cropped = resized.crop((left, top, left + crop, top + crop))

# RGB → Normalize → NCHW
pixels = np.asarray(cropped, dtype=np.float32) / 255.0
mean = np.array(config['mean'], dtype=np.float32)
std = np.array(config['std'], dtype=np.float32)
onnx_input = np.ascontiguousarray(((pixels - mean) / std).transpose(2, 0, 1)[None])  # HWC → CHW → NCHW

session = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])  # โหลด Model โดยไม่ใช้ PyTorch/CUDA
onnx_logits = session.run(['logits'], {'images': onnx_input})[0]  # [0] เลือก Output แรก ได้ Shape [1,3]
scores = np.exp(onnx_logits[0] - onnx_logits[0].max())  # Stable Softmax: ลบค่าสูงสุดเพื่อลด Overflow
scores = scores / scores.sum()
print('Prediction:', config['class_names'][int(scores.argmax())])
print(dict(zip(config['class_names'], scores.tolist())))
display(image)

### การตรวจสอบ Output Parity ระหว่าง ONNX และ PyTorch

เปรียบเทียบ Input Tensor หลัง Preprocessing และค่า Logits ภายใต้ Tolerance ที่กำหนด


In [ ]:
reference_input = selected_transform(image).unsqueeze(0)
np.testing.assert_allclose(onnx_input, reference_input.numpy(), atol=1e-6, rtol=0)  # ตรวจ Preprocessing Parity
with torch.no_grad():
    pytorch_logits = loaded_model(reference_input).numpy()
np.testing.assert_allclose(onnx_logits, pytorch_logits, atol=1e-4, rtol=1e-3)  # ตรวจ Logits ไม่ใช่เพียง Class ที่ทายตรงกัน
print('ONNX / PyTorch OK')

## การดาวน์โหลดผลลัพธ์

ดาวน์โหลดไฟล์ ONNX และ JSON ไว้ด้วยกันก่อน Colab ล้างข้อมูลใน Runtime


In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    files.download(shutil.make_archive('/content/food_transfer_results', 'zip', OUT_DIR))

ResNet18, EfficientNet-B0 และ MobileNetV3-Small ใช้ ImageNet Pretrained Weights และฝึกเฉพาะ Classification Head
ส่วน Custom CNN ใช้ Random Initialization และฝึกทุก Layer
ใช้ Validation Set สำหรับ Model Selection และสงวน Test Set สำหรับ Final Evaluation
Model จำแนกภาพเป็นหนึ่งในสาม Classes เสมอ แม้ภาพนั้นไม่ใช่อาหาร
ตัวอย่าง ONNX นี้รองรับ Inference ในเครื่อง ยังไม่ครอบคลุมการให้บริการผ่าน Production API
